# Advanced QAOA for Constrained Portfolio Optimization

## 1. Project Overview

This notebook implements a constrained portfolio optimization problem using the **Quantum Approximate Optimization Algorithm (QAOA)**. The experiment combines classical optimization, quantum circuit simulation, and execution on IBM quantum hardware.

The goal is to select 10 stocks from a synthetic universe of 20 stocks while satisfying sector constraints and minimizing a risk-return objective. An exhaustive classical search provides the reference optimum against which the QAOA samples are evaluated.

This project is an experimental demonstration of quantum optimization. It does not claim quantum advantage over classical methods.

## 2. Problem Statement

An investment portfolio must balance expected returns against risk while respecting portfolio composition constraints.

The problem in this notebook has the following configuration:

| Parameter | Value |
|---|---:|
| Total stocks | 20 |
| Stocks to select | 10 |
| Number of sectors | 5 |
| Stocks in each sector | 4 |
| Maximum stocks per sector | 2 |
| Risk-aversion parameter | 0.5 |
| Cardinality penalty | 5.0 |
| Sector-constraint penalty | 5.0 |
| Random seed | 2026 |

Each stock is represented by a binary decision variable:

- \(x_i=1\): stock \(i\) is selected.
- \(x_i=0\): stock \(i\) is not selected.

The resulting decision vector is \(x \in \{0,1\}^{20}\).

## 3. Synthetic Dataset

The experiment generates synthetic expected returns and a covariance matrix using NumPy with a fixed random seed of 2026.

Expected returns are generated from a uniform distribution between 0.05 and 0.25. A random matrix is used to construct the covariance matrix:

\[
\Sigma = AA^T
\]

This construction produces a positive-semidefinite covariance matrix, which is suitable for modeling portfolio risk.

The data are synthetic rather than historical market observations. Therefore, the experiment evaluates the optimization method and its constraints, not the profitability of real investments.

## 4. Mathematical Formulation

### 4.1 Portfolio risk

Portfolio risk is modeled as:

\[
R(x)=x^T\Sigma x
\]

Here, \(\Sigma\) is the covariance matrix and \(x\) is the binary selection vector.

### 4.2 Expected return

The expected portfolio return is represented by:

\[
M(x)=\mu^T x
\]

where \(\mu\) contains the expected return of each stock.

### 4.3 Risk-return objective

The unpenalized objective is:

\[
C(x)=q\,x^T\Sigma x-\mu^T x
\]

where \(q=0.5\) controls the trade-off between risk and expected return.

A lower objective value is preferred. The negative return term rewards portfolios with greater expected return, while the risk term penalizes higher portfolio risk.

### 4.4 Cardinality constraint

Exactly 10 stocks must be selected:

\[
\sum_{i=1}^{20}x_i=10
\]

This constraint is encouraged by adding a quadratic penalty:

\[
P_{\mathrm{card}}(x)=
\lambda_{\mathrm{card}}
\left(\sum_{i=1}^{20}x_i-10\right)^2
\]

### 4.5 Sector constraint

The 20 stocks are divided into five sectors of four stocks each. No sector may contain more than two selected stocks.

For sector \(s\), let \(n_s\) denote its selected-stock count. The implementation penalizes deviations from two selected stocks per sector:

\[
P_{\mathrm{sector}}(x)=
\lambda_{\mathrm{sector}}
\sum_{s=1}^{5}(n_s-2)^2
\]

The target of two stocks per sector is consistent with the cardinality constraint: selecting 10 stocks across five sectors, with a maximum of two from each sector, requires exactly two per sector.

### 4.6 Combined objective

The cost function used for the QAOA formulation is:

\[
\begin{aligned}
C_{\mathrm{total}}(x)
={}&q\,x^T\Sigma x-\mu^T x\\
&+\lambda_{\mathrm{card}}
\left(\sum_i x_i-10\right)^2\\
&+\lambda_{\mathrm{sector}}
\sum_{s=1}^{5}(n_s-2)^2
\end{aligned}
\]

The penalty coefficients are both 5.0.

The penalty terms discourage infeasible selections. For the final portfolio comparison, the unpenalized risk-return objective is also evaluated separately.

## 5. Classical Reference Solution

Before evaluating QAOA, the notebook enumerates all feasible portfolios.

Because each of the five sectors contains four stocks and exactly two stocks must be selected from each sector, the number of feasible portfolios is:

\[
\binom{4}{2}^{5}=6^5=7,776
\]

The notebook evaluates every feasible portfolio and identifies the minimum-cost solution. This provides an exact classical reference for the defined synthetic instance.

**Classical reference optimum:**

- Selected stocks: Stock 2, Stock 4, Stock 5, Stock 7, Stock 11, Stock 12, Stock 13, Stock 16, Stock 18, and Stock 20.
- Expected return score: 1.736840.
- Portfolio risk: 1.250202.
- Minimum unpenalized objective cost: −1.111739.

This exhaustive search is computationally practical for the current problem size and serves as the baseline for evaluating the quantum results.

## 6. QUBO and Ising Hamiltonian

QAOA operates on a quantum cost Hamiltonian. The binary optimization problem is therefore converted into a form that can be represented using qubits and Pauli operators.

The implementation constructs a Hamiltonian from the objective and its constraint penalties. Its diagonal energy encodes the cost assigned to computational-basis bitstrings.

The notebook's Hamiltonian contains:

- 20 qubits, corresponding to 20 binary stock-selection variables.
- 211 Pauli terms.
- A numerical consistency check against the original classical cost function.

The consistency check evaluated 101 bitstrings, with a maximum absolute error of approximately \(5.116\times10^{-13}\). This supports the numerical correctness of the tested Hamiltonian encoding.

## 7. QAOA Circuit Design

The notebook implements QAOA with circuit depth parameter \(p=1\).

The circuit contains an initial superposition, problem

## Implementation Walkthrough

The following sections implement and evaluate the constrained portfolio optimization problem described above. The workflow progresses from generating reproducible synthetic data and establishing an exact classical reference solution to constructing the QAOA cost Hamiltonian, simulating the quantum circuit, and evaluating its performance on IBM quantum hardware.

Each stage reports its key assumptions and results so that the implementation can be followed and reproduced.

### Step 1: Generate the Portfolio Dataset

We begin by creating a reproducible synthetic investment universe containing 20 stocks. Each stock has an expected-return value, and the portfolio risk is represented by a covariance matrix.

The dataset uses NumPy's random seed of 2026 so that the synthetic problem can be regenerated consistently using the same data-generation procedure.

The portfolio is configured to select exactly 10 stocks, with no more than two stocks from each of five sectors. The risk-aversion parameter is \(q=0.5\).

**Purpose:** Establish a controlled optimization problem before introducing the quantum algorithm.

**Important:** These are synthetic data, not historical stock-market observations. They are used to evaluate the optimization workflow rather than to make financial predictions.

### Step 2: Establish the Exact Classical Baseline

Before evaluating QAOA, we solve the constrained problem using exhaustive classical search.

Each sector contains four stocks, and exactly two stocks must be selected from each sector. Therefore, the number of feasible portfolios is

\[
\binom{4}{2}^{5}=6^5=7,776.
\]

We evaluate every feasible portfolio using the unpenalized risk-return objective and retain the lowest-cost solution.

This exhaustive search provides an exact reference optimum for the defined synthetic instance. It allows us to measure how closely the simulator and hardware solutions approach the best known feasible portfolio.

**Why this matters:** Finding a good-looking bitstring is not enough. We need a reliable classical reference to evaluate solution quality.

### Step 3: Define the Constrained Optimization Objective

The portfolio objective balances risk against expected return:

\[
C(x)=q\,x^T\Sigma x-\mu^T x,
\]

where \(x\) is the binary stock-selection vector, \(\Sigma\) is the covariance matrix, \(\mu\) contains expected returns, and \(q\) controls risk aversion.

Two constraints must also be satisfied:

1. Exactly 10 stocks must be selected.
2. Each sector may contribute no more than two stocks.

The implementation introduces quadratic penalty terms for violations of the required portfolio size and sector composition. These penalties are added to the objective used by QAOA.

For the final portfolio comparison, we separately evaluate the unpenalized risk-return objective. This distinction matters because a low penalized cost does not, by itself, establish that a portfolio is feasible or financially preferable.

In [7]:
# Track 2 — Advanced QAOA
# Step 1: Generate a 20-stock problem with sector constraints

import numpy as np
from itertools import combinations, product
from math import comb

# Reproducible challenge-style dataset
rng = np.random.default_rng(2026)

n = 20
K = 10
q = 0.5

num_sectors = 5
stocks_per_sector = 4
sector_cap = 2

mu = rng.uniform(0.05, 0.25, n)

A = rng.normal(0, 0.1, (n, n))
sigma = A @ A.T

# Each sector contains four stocks.
sectors = [
    list(range(s * stocks_per_sector,
               (s + 1) * stocks_per_sector))
    for s in range(num_sectors)
]

# Build every legal choice of two stocks from each sector.
sector_choices = [
    list(combinations(sector, sector_cap))
    for sector in sectors
]

valid_portfolios = []

for choices in product(*sector_choices):
    x = np.zeros(n, dtype=np.int8)

    for selected_pair in choices:
        x[list(selected_pair)] = 1

    valid_portfolios.append(x)

X_valid = np.asarray(valid_portfolios, dtype=np.int8)
X_float = X_valid.astype(float)

# Calculate the original return-risk objective
# for every feasible portfolio.
expected_returns = X_float @ mu

risks = np.einsum(
    "bi,ij,bj->b",
    X_float,
    sigma,
    X_float,
    optimize=True
)

costs = q * risks - expected_returns

# Exact classical solution over all feasible portfolios
best_row = int(np.argmin(costs))
best_x = X_valid[best_row].copy()
best_cost = float(costs[best_row])

best_stocks = [
    f"Stock {i + 1}"
    for i, selected in enumerate(best_x)
    if selected
]

# Validate all constraints
sector_counts = np.column_stack([
    X_valid[:, sector].sum(axis=1)
    for sector in sectors
])

assert X_valid.shape == (
    comb(stocks_per_sector, sector_cap) ** num_sectors,
    n
)

assert np.all(X_valid.sum(axis=1) == K)
assert np.all(sector_counts <= sector_cap)
assert np.allclose(sigma, sigma.T)

# Verify that the covariance matrix is positive semidefinite
assert np.linalg.eigvalsh(sigma).min() >= -1e-10

print("=" * 58)
print("ADVANCED QAOA — CLASSICAL REFERENCE")
print("=" * 58)

print(f"Number of stocks: {n}")
print(f"Stocks to select: {K}")
print(f"Number of sectors: {num_sectors}")
print(f"Stocks per sector: {stocks_per_sector}")
print(f"Sector selection limit: {sector_cap}")

print(f"\nPortfolios before sector constraints: {comb(n, K):,}")
print(f"Feasible portfolios: {len(X_valid):,}")

print("\nExpected returns:")
for i, value in enumerate(mu):
    print(f"Stock {i + 1:02d}: {value:.5f}")

print("\nSector allocation:")
for s, sector in enumerate(sectors):
    print(
        f"Sector {s + 1}: "
        f"{[f'Stock {i + 1}' for i in sector]}"
    )

print("\nExact classical optimum:")
print("Selected stocks:", best_stocks)
print(f"Expected return: {mu @ best_x:.6f}")
print(f"Portfolio risk: {best_x @ sigma @ best_x:.6f}")
print(f"Minimum feasible cost: {best_cost:.6f}")

print("\nAll classical feasibility checks passed.")

ADVANCED QAOA — CLASSICAL REFERENCE
Number of stocks: 20
Stocks to select: 10
Number of sectors: 5
Stocks per sector: 4
Sector selection limit: 2

Portfolios before sector constraints: 184,756
Feasible portfolios: 7,776

Expected returns:
Stock 01: 0.08579
Stock 02: 0.17798
Stock 03: 0.14345
Stock 04: 0.12410
Stock 05: 0.12098
Stock 06: 0.20810
Stock 07: 0.23103
Stock 08: 0.08547
Stock 09: 0.18056
Stock 10: 0.10966
Stock 11: 0.24339
Stock 12: 0.23397
Stock 13: 0.17717
Stock 14: 0.20055
Stock 15: 0.15303
Stock 16: 0.21518
Stock 17: 0.13968
Stock 18: 0.11776
Stock 19: 0.10558
Stock 20: 0.09527

Sector allocation:
Sector 1: ['Stock 1', 'Stock 2', 'Stock 3', 'Stock 4']
Sector 2: ['Stock 5', 'Stock 6', 'Stock 7', 'Stock 8']
Sector 3: ['Stock 9', 'Stock 10', 'Stock 11', 'Stock 12']
Sector 4: ['Stock 13', 'Stock 14', 'Stock 15', 'Stock 16']
Sector 5: ['Stock 17', 'Stock 18', 'Stock 19', 'Stock 20']

Exact classical optimum:
Selected stocks: ['Stock 2', 'Stock 4', 'Stock 5', 'Stock 7', 'Stock 

In [8]:
%pip install -q qiskit

### Step 4: Construct the QAOA Cost Hamiltonian

QAOA requires the classical objective to be encoded as a quantum cost Hamiltonian.

The binary decision variables are mapped to qubits, and the cost function is transformed into a representation involving Pauli operators. The resulting Hamiltonian assigns energies to computational-basis states according to the encoded optimization objective.

For this problem, the Hamiltonian uses 20 logical qubits, corresponding to the 20 stock-selection variables.

The correctness of this transformation is important: if the Hamiltonian does not represent the intended classical cost, the quantum circuit will optimize the wrong problem.

The following validation step checks the encoded energy against the original classical objective.

### Step 5: Validate the Hamiltonian Encoding

Before running QAOA, we compare the energy produced by the encoded Hamiltonian with the classical cost calculated directly from the original objective.

The validation checks 101 bitstrings and reports a maximum absolute error of approximately \(5.116\times10^{-13}\).

The very small numerical discrepancy supports the correctness of the Hamiltonian encoding for the tested inputs, subject to the coverage of this validation.

This check is essential because optimization results are meaningful only when the quantum representation matches the intended classical problem.

In [9]:
# Advanced QAOA — Step 2
# Construct and validate the constrained cost Hamiltonian

import numpy as np
from qiskit.quantum_info import SparsePauliOp

# Penalty strengths
lambda_card = 5.0
lambda_sector = 5.0


def advanced_cost(x):
    """Portfolio cost including cardinality and sector penalties."""
    x = np.asarray(x, dtype=int)

    total_selected = int(x.sum())

    sector_counts = np.array([
        x[sector].sum() for sector in sectors
    ])

    return float(
        q * (x @ sigma @ x)
        - (mu @ x)
        + lambda_card * (total_selected - K) ** 2
        + lambda_sector * np.sum(
            (sector_counts - sector_cap) ** 2
        )
    )


# Identify the sector membership of every stock
stock_sector = {
    stock: sector_id
    for sector_id, sector in enumerate(sectors)
    for stock in sector
}

# Expand the objective into:
# constant + sum(linear[i] * x[i])
#           + sum(pair[i,j] * x[i] * x[j])

linear = (
    q * np.diag(sigma)
    - mu
    + lambda_card * (1 - 2 * K)
    + lambda_sector * (1 - 2 * sector_cap)
)

pair = {}

for i in range(n):
    for j in range(i + 1, n):
        coefficient = (
            2 * q * sigma[i, j]
            + 2 * lambda_card
        )

        # Sector penalty adds an interaction within each sector
        if stock_sector[i] == stock_sector[j]:
            coefficient += 2 * lambda_sector

        pair[(i, j)] = float(coefficient)


# Constant contributions from the squared penalties
qubo_constant = (
    lambda_card * K**2
    + lambda_sector * len(sectors) * sector_cap**2
)

# Convert binary variables x_i into Pauli Z variables:
# x_i = (1 - Z_i) / 2

identity_coefficient = (
    qubo_constant
    + 0.5 * np.sum(linear)
    + 0.25 * sum(pair.values())
)

pauli_terms = [
    ("I" * n, float(identity_coefficient))
]

# Single-qubit Z terms
for i in range(n):
    pair_sum = sum(
        coefficient
        for (a, b), coefficient in pair.items()
        if a == i or b == i
    )

    z_coefficient = (
        -0.5 * linear[i]
        - 0.25 * pair_sum
    )

    label = ["I"] * n
    label[n - 1 - i] = "Z"

    pauli_terms.append((
        "".join(label),
        float(z_coefficient)
    ))

# Two-qubit ZZ terms
for (i, j), coefficient in pair.items():
    label = ["I"] * n
    label[n - 1 - i] = "Z"
    label[n - 1 - j] = "Z"

    pauli_terms.append((
        "".join(label),
        float(coefficient / 4)
    ))

cost_hamiltonian_20 = (
    SparsePauliOp.from_list(pauli_terms).simplify()
)


# Validate the Hamiltonian on computational-basis bitstrings.
# For a bitstring x, Z_i has eigenvalue 1 - 2*x_i.

def pauli_energy_for_bitstring(x):
    z_values = 1.0 - 2.0 * np.asarray(x, dtype=float)
    energy = 0.0

    for label, coefficient in cost_hamiltonian_20.to_list():
        eigenvalue = 1.0

        for position, symbol in enumerate(label):
            if symbol == "Z":
                qubit = n - 1 - position
                eigenvalue *= z_values[qubit]
            elif symbol != "I":
                raise ValueError(
                    f"Unexpected non-diagonal Pauli: {symbol}"
                )

        energy += float(np.real(coefficient)) * eigenvalue

    return float(energy)


# Test the exact feasible optimum and 100 random bitstrings.
validation_rng = np.random.default_rng(20261014)

test_bitstrings = [best_x.copy()]

test_bitstrings.extend(
    validation_rng.integers(0, 2, size=(100, n))
)

errors = []

for x in test_bitstrings:
    classical_value = advanced_cost(x)
    pauli_value = pauli_energy_for_bitstring(x)

    errors.append(abs(classical_value - pauli_value))

max_error = max(errors)

assert max_error < 1e-8, (
    f"Hamiltonian validation failed; maximum error: {max_error}"
)

# Check that the feasible optimum retains its original objective value.
assert np.isclose(
    advanced_cost(best_x),
    best_cost,
    atol=1e-8
)


print("=" * 58)
print("ADVANCED QAOA — HAMILTONIAN VALIDATION")
print("=" * 58)

print(f"Qubits: {n}")
print(f"Cardinality penalty: {lambda_card}")
print(f"Sector penalty: {lambda_sector}")
print(f"Pauli terms: {len(cost_hamiltonian_20)}")

print("\nExact feasible optimum:")
print("Selected stocks:", best_stocks)
print(f"Classical cost: {advanced_cost(best_x):.6f}")
print(
    f"Hamiltonian cost: "
    f"{pauli_energy_for_bitstring(best_x):.6f}"
)

print(f"\nBitstrings checked: {len(test_bitstrings)}")
print(f"Maximum numerical error: {max_error:.3e}")
print("\nHamiltonian validation passed.")

ADVANCED QAOA — HAMILTONIAN VALIDATION
Qubits: 20
Cardinality penalty: 5.0
Sector penalty: 5.0
Pauli terms: 211

Exact feasible optimum:
Selected stocks: ['Stock 2', 'Stock 4', 'Stock 5', 'Stock 7', 'Stock 11', 'Stock 12', 'Stock 13', 'Stock 16', 'Stock 18', 'Stock 20']
Classical cost: -1.111739
Hamiltonian cost: -1.111739

Bitstrings checked: 101
Maximum numerical error: 5.116e-13

Hamiltonian validation passed.


### Step 6: Design the QAOA Circuit

The Quantum Approximate Optimization Algorithm alternates between problem-dependent cost evolution and a mixing operation. The initial state places the qubits into a superposition of candidate bitstrings.

The circuit in this notebook uses \(p=1\), giving two trainable angles: \(\gamma\) for cost evolution and \(\beta\) for mixing.

For the 20-stock instance, the circuit has 20 logical qubits and two trainable parameters. After its construction, the circuit can be parameterized, optimized, simulated, and prepared for hardware execution.

The QAOA circuit samples candidate solutions probabilistically. Its measurements must be decoded and checked against the portfolio constraints before the resulting portfolios can be evaluated.

In [10]:

# Advanced QAOA
# Build and inspect a 20-qubit QAOA circuit

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager

p = 1

gamma = ParameterVector("gamma", p)
beta = ParameterVector("beta", p)

qc_advanced = QuantumCircuit(n)

# Initial equal superposition
qc_advanced.h(range(n))

# Cost unitary: exp(-i * gamma * H_C)
# Our Hamiltonian contains only I, Z, and ZZ terms.
for layer in range(p):
    for label, coefficient in cost_hamiltonian_20.to_list():
        coefficient = float(np.real(coefficient))

        z_qubits = [
            n - 1 - position
            for position, symbol in enumerate(label)
            if symbol == "Z"
        ]

        # Identity terms contribute only a global phase.
        if len(z_qubits) == 1:
            qc_advanced.rz(
                2 * coefficient * gamma[layer],
                z_qubits[0]
            )

        elif len(z_qubits) == 2:
            i, j = z_qubits
            qc_advanced.rzz(
                2 * coefficient * gamma[layer],
                i, j
            )

    # Standard X-mixer unitary
    for qubit in range(n):
        qc_advanced.rx(2 * beta[layer], qubit)

print("=" * 58)
print("ADVANCED QAOA — CIRCUIT CHECK")
print("=" * 58)

print("Number of qubits:", qc_advanced.num_qubits)
print("QAOA depth parameter p:", p)
print("Unbound parameters:", qc_advanced.parameters)
print("Circuit depth:", qc_advanced.depth())
print("Circuit operations:", qc_advanced.count_ops())

assert qc_advanced.num_qubits == 20
assert len(qc_advanced.parameters) == 2 * p
assert qc_advanced.depth() > 0

print("\nCircuit construction and checks passed.")


ADVANCED QAOA — CIRCUIT CHECK
Number of qubits: 20
QAOA depth parameter p: 1
Unbound parameters: ParameterView([ParameterVectorElement(beta[0]), ParameterVectorElement(gamma[0])])
Circuit depth: 40
Circuit operations: OrderedDict({'rzz': 190, 'h': 20, 'rz': 20, 'rx': 20})

Circuit construction and checks passed.


In [ ]:
%pip -q install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 97.3 MB/s eta 0:00:00


In [ ]:

# Advanced QAOA
# Set up a shot-based simulator and parameterized QAOA circuit

from qiskit import transpile
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

# Use a shot-based simulator to estimate the objective.
simulator = AerSimulator()

# Add measurement to a copy of the circuit.
qc_measured = qc_advanced.copy()
qc_measured.measure_all()

# Bind one candidate pair of QAOA parameters.
def make_bound_circuit(gamma_value, beta_value):
    parameter_values = {
        gamma[0]: float(gamma_value),
        beta[0]: float(beta_value),
    }

    return qc_measured.assign_parameters(
        parameter_values,
        inplace=False
    )

# Verify that parameter binding works before optimization.
test_circuit = make_bound_circuit(0.5, 0.25)

assert len(test_circuit.parameters) == 0
assert test_circuit.num_qubits == 20
assert test_circuit.num_clbits == 20

print("Simulator:", simulator.name)
print("Measured qubits:", test_circuit.num_qubits)
print("Classical measurement bits:", test_circuit.num_clbits)
print("Unbound parameters:", len(test_circuit.parameters))
print("\nParameter binding validation passed.")


Simulator: aer_simulator
Measured qubits: 20
Classical measurement bits: 20
Unbound parameters: 0

Parameter binding validation passed.


### Step 7: Perform a Pilot Parameter Optimization

QAOA's circuit parameters determine how the quantum state evolves before measurement. A classical optimizer searches for parameter values that improve the chosen objective.

This notebook performs a limited pilot optimization using a maximum of 20 function evaluations. The search reached that evaluation limit, so the resulting parameters are treated as pilot values rather than a proven converged optimum.

The pilot values used in the subsequent hardware experiment were approximately \(\gamma=0.4625\) and \(\beta=0.25625\).

This limitation is important when interpreting the results. A more extensive search, multiple initializations, or alternative optimization methods could produce different outcomes, but improvements must be demonstrated experimentally.

In [ ]:

# Advanced QAOA
# Optimize p=1 using sampled measurements

from scipy.optimize import minimize
from collections import Counter
import time

SHOTS = 512
MAX_EVALUATIONS = 20

# Compile once so repeated evaluations are faster.
compiled_circuit = transpile(
    qc_measured,
    simulator,
    optimization_level=1,
    seed_transpiler=42
)

evaluation_count = 0
optimization_history = []
start_time = time.time()


def evaluate_qaoa(params):
    global evaluation_count

    gamma_value, beta_value = params

    bound = compiled_circuit.assign_parameters({
        gamma[0]: float(gamma_value),
        beta[0]: float(beta_value),
    })

    job = simulator.run(
        bound,
        shots=SHOTS,
        seed_simulator=1000 + evaluation_count
    )

    counts = job.result().get_counts()

    estimated_cost = 0.0

    for bitstring, count in counts.items():
        x = np.array(
            [int(bit) for bit in bitstring.replace(" ", "")[::-1]],
            dtype=int
        )

        estimated_cost += (
            advanced_cost(x) * count / SHOTS
        )

    evaluation_count += 1
    optimization_history.append(
        (gamma_value, beta_value, estimated_cost)
    )

    print(
        f"Evaluation {evaluation_count:02d}: "
        f"gamma={gamma_value:.4f}, "
        f"beta={beta_value:.4f}, "
        f"estimated cost={estimated_cost:.4f}"
    )

    return estimated_cost


# Constrain the parameters to a reasonable initial search range.
result = minimize(
    evaluate_qaoa,
    x0=np.array([0.5, 0.25]),
    method="Nelder-Mead",
    bounds=[(0.0, np.pi), (0.0, np.pi / 2)],
    options={
        "maxfev": MAX_EVALUATIONS,
        "xatol": 0.05,
        "fatol": 0.05,
    }
)

elapsed = time.time() - start_time

print("\n" + "=" * 58)
print("QAOA PILOT OPTIMIZATION SUMMARY")
print("=" * 58)
print("Evaluations:", evaluation_count)
print("Best parameters:", result.x)
print("Best estimated expected cost:", result.fun)
print(f"Elapsed time: {elapsed:.2f} seconds")
print("Optimizer status:", result.message)


Evaluation 01: gamma=0.5000, beta=0.2500, estimated cost=83.6034
Evaluation 02: gamma=0.5250, beta=0.2500, estimated cost=84.3226
Evaluation 03: gamma=0.5000, beta=0.2625, estimated cost=79.7683
Evaluation 04: gamma=0.4750, beta=0.2625, estimated cost=68.9264
Evaluation 05: gamma=0.4500, beta=0.2687, estimated cost=72.5099
Evaluation 06: gamma=0.4750, beta=0.2750, estimated cost=79.6245
Evaluation 07: gamma=0.4500, beta=0.2750, estimated cost=74.9115
Evaluation 08: gamma=0.4500, beta=0.2625, estimated cost=72.5723
Evaluation 09: gamma=0.4750, beta=0.2500, estimated cost=68.1078
Evaluation 10: gamma=0.4875, beta=0.2375, estimated cost=74.3205
Evaluation 11: gamma=0.5000, beta=0.2500, estimated cost=79.8355
Evaluation 12: gamma=0.4625, beta=0.2594, estimated cost=73.9956
Evaluation 13: gamma=0.4750, beta=0.2562, estimated cost=73.4702
Evaluation 14: gamma=0.4625, beta=0.2562, estimated cost=67.7462
Evaluation 15: gamma=0.4625, beta=0.2500, estimated cost=72.2650
Evaluation 16: gamma=0.46

### Step 8: Evaluate QAOA on the Simulator

We first evaluate the parameterized QAOA circuit in simulation before examining the hardware results.

The simulator produces measurement counts for 4,096 shots. Each measured bitstring is converted into a portfolio-selection vector, and the selected stocks are checked against the cardinality and sector constraints.

The analysis records the number of valid portfolio samples, the number of times the exact classical optimum appears, and the objective values of valid solutions.

The exact classical optimum was observed once in the simulator results. Although this demonstrates that the circuit sampled the optimum, one occurrence does not establish reliable optimization performance or quantum advantage.

In [ ]:

# Advanced QAOA
# Evaluate optimized parameters on fresh measurements

from collections import Counter
import numpy as np

EVAL_SHOTS = 4096

gamma_best, beta_best = result.x

final_circuit = compiled_circuit.assign_parameters({
    gamma[0]: float(gamma_best),
    beta[0]: float(beta_best),
})

evaluation_job = simulator.run(
    final_circuit,
    shots=EVAL_SHOTS,
    seed_simulator=20261014
)

final_counts = evaluation_job.result().get_counts()

valid_shots = 0
exact_optimum_shots = 0
best_sampled_cost = float("inf")
best_sampled_x = None

base_cost_sum = 0.0
penalized_cost_sum = 0.0
sampled_portfolios = Counter()

for bitstring, count in final_counts.items():
    # Qiskit displays classical bit 0 on the right.
    x = np.array(
        [int(bit) for bit in bitstring.replace(" ", "")[::-1]],
        dtype=int
    )

    total_selected = int(x.sum())

    sector_counts_sample = np.array([
        int(x[sector].sum())
        for sector in sectors
    ])

    is_valid = (
        total_selected == K
        and np.all(sector_counts_sample <= sector_cap)
    )

    base_cost = float(
        q * (x @ sigma @ x) - (mu @ x)
    )

    penalized_cost = advanced_cost(x)

    base_cost_sum += count * base_cost
    penalized_cost_sum += count * penalized_cost

    sampled_portfolios[tuple(x)] += count

    if is_valid:
        valid_shots += count

        if base_cost < best_sampled_cost:
            best_sampled_cost = base_cost
            best_sampled_x = x.copy()

        if np.array_equal(x, best_x):
            exact_optimum_shots += count

valid_percentage = 100 * valid_shots / EVAL_SHOTS
exact_optimum_percentage = (
    100 * exact_optimum_shots / EVAL_SHOTS
)

print("=" * 60)
print("ADVANCED QAOA — FINAL SAMPLE EVALUATION")
print("=" * 60)

print("Fresh measurement shots:", EVAL_SHOTS)
print(f"Optimized gamma: {gamma_best:.6f}")
print(f"Optimized beta: {beta_best:.6f}")

print(
    f"\nValid portfolios: {valid_shots}/{EVAL_SHOTS} "
    f"({valid_percentage:.2f}%)"
)

print(
    f"Exact classical optimum sampled: "
    f"{exact_optimum_shots}/{EVAL_SHOTS} "
    f"({exact_optimum_percentage:.3f}%)"
)

print(
    "\nMean unpenalized objective:",
    f"{base_cost_sum / EVAL_SHOTS:.6f}"
)

print(
    "Mean penalized objective:",
    f"{penalized_cost_sum / EVAL_SHOTS:.6f}"
)

if best_sampled_x is not None:
    selected_stocks = [
        f"Stock {i + 1}"
        for i, bit in enumerate(best_sampled_x)
        if bit
    ]

    print("\nBest valid sampled portfolio:")
    print("Selected stocks:", selected_stocks)
    print(f"Portfolio cost: {best_sampled_cost:.6f}")
    print(f"Classical optimum cost: {best_cost:.6f}")

    print(
        "Absolute cost gap:",
        f"{best_sampled_cost - best_cost:.6f}"
    )

    print("\nSector selections:")
    for s, sector in enumerate(sectors):
        chosen = [
            f"Stock {i + 1}"
            for i in sector
            if best_sampled_x[i]
        ]
        print(f"Sector {s + 1}: {chosen}")

else:
    print("\nNo valid portfolio was sampled.")
    print("We will need to improve constraint handling.")

print("\nEvaluation complete.")


ADVANCED QAOA — FINAL SAMPLE EVALUATION
Fresh measurement shots: 4096
Optimized gamma: 0.462500
Optimized beta: 0.256250

Valid portfolios: 422/4096 (10.30%)
Exact classical optimum sampled: 1/4096 (0.024%)

Mean unpenalized objective: -0.348625
Mean penalized objective: 73.574714

Best valid sampled portfolio:
Selected stocks: ['Stock 2', 'Stock 4', 'Stock 5', 'Stock 7', 'Stock 11', 'Stock 12', 'Stock 13', 'Stock 16', 'Stock 18', 'Stock 20']
Portfolio cost: -1.111739
Classical optimum cost: -1.111739
Absolute cost gap: 0.000000

Sector selections:
Sector 1: ['Stock 2', 'Stock 4']
Sector 2: ['Stock 5', 'Stock 7']
Sector 3: ['Stock 11', 'Stock 12']
Sector 4: ['Stock 13', 'Stock 16']
Sector 5: ['Stock 18', 'Stock 20']

Evaluation complete.


In [2]:
%pip install -q qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541

In [4]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)
print("IBM Runtime:", qiskit_ibm_runtime.__version__)
print("Environment ready!")

Qiskit: 2.5.2
Qiskit Aer: 0.17.2
IBM Runtime: 0.50.0
Environment ready!


In [3]:
import importlib.util

print("PACKAGE CHECK")
for package in ["qiskit", "qiskit_aer", "qiskit_ibm_runtime", "scipy"]:
    print(f"{package}:",
          "Installed" if importlib.util.find_spec(package) else "Missing")

print("\nADVANCED QAOA VARIABLES")
for name in [
    "mu", "sigma", "advanced_cost", "best_x",
    "cost_hamiltonian_20", "qc_advanced",
    "qc_measured", "compiled_circuit", "result"
]:
    print(f"{name}:",
          "Available" if name in globals() else "Missing")

PACKAGE CHECK
qiskit: Installed
qiskit_aer: Installed
qiskit_ibm_runtime: Installed
scipy: Installed

ADVANCED QAOA VARIABLES
mu: Missing
sigma: Missing
advanced_cost: Missing
best_x: Missing
cost_hamiltonian_20: Missing
qc_advanced: Missing
qc_measured: Missing
compiled_circuit: Missing
result: Missing


### Step 9: Prepare the Circuit for IBM Quantum Hardware

After the simulator evaluation, we prepare the parameter-bound QAOA circuit for execution on IBM quantum hardware.

The notebook connects through IBM Quantum Runtime, selects the `ibm_kingston` backend, adds measurement operations, and transpiles the circuit into instructions supported by the device.

Transpilation may change the circuit's width, depth, and gate count to accommodate the backend's connectivity and native operations.

For this experiment, the compiled circuit used 156 physical-qubit positions, had a depth of 680, and contained 938 CZ gates. The logical optimization problem itself still contains 20 decision qubits.

Because deep circuits can be sensitive to accumulated gate and readout errors, these hardware characteristics are relevant when interpreting the measured results.

**Security note:** IBM credentials are retrieved from Colab Secrets. API keys should never be pasted directly into the notebook or committed to the public repository.

In [11]:
from qiskit import transpile
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

# Reconnect to IBM Quantum
api_key = userdata.get("IBM_QUANTUM_API_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=api_key,
)
backend = service.backend("ibm_kingston")

# Exploratory parameters from the earlier limited optimization pilot.
# The pilot had not converged.
gamma_test = 0.4625
beta_test = 0.25625

# Bind by parameter name to avoid beta/gamma ordering mistakes.
binding = {}
for parameter in qc_advanced.parameters:
    if "gamma" in parameter.name.lower():
        binding[parameter] = gamma_test
    elif "beta" in parameter.name.lower():
        binding[parameter] = beta_test
    else:
        raise ValueError(f"Unexpected QAOA parameter: {parameter}")

qaoa_bound_adv = qc_advanced.assign_parameters(binding)

# Add measurements and compile for the selected QPU.
qaoa_measured_adv = qaoa_bound_adv.copy()
qaoa_measured_adv.measure_all()

print("Backend:", backend.name)
print("Operational:", backend.status().operational)
print("Logical qubits:", qc_advanced.num_qubits)
print("Unbound parameters:", qaoa_measured_adv.num_parameters)

qaoa_isa_adv = transpile(
    qaoa_measured_adv,
    backend=backend,
    optimization_level=1,
)

print("\nCompilation complete.")
print("Compiled circuit width:", qaoa_isa_adv.num_qubits)
print("Compiled circuit depth:", qaoa_isa_adv.depth())
print("Compiled operation counts:", qaoa_isa_adv.count_ops())

qiskit_runtime_service._discover_account:WARNING:2026-10-14 12:24:41,547: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 12:24:44,716: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 12:24:44,721: Using instance: open-instance, plan: open


Backend: ibm_kingston
Operational: True
Logical qubits: 20
Unbound parameters: 0

Compilation complete.
Compiled circuit width: 156
Compiled circuit depth: 680
Compiled operation counts: OrderedDict({'sx': 1701, 'rz': 1336, 'cz': 938, 'measure': 20, 'barrier': 1})


### Step 10: Execute QAOA on Real Quantum Hardware

The parameter-bound circuit is submitted to IBM Quantum using 256 measurement shots. The returned measurement counts are retained for offline analysis.

This is an actual quantum-hardware execution, distinct from the earlier simulator experiment. The resulting counts reflect the behavior of the compiled circuit on the selected backend, including the effects of real-device noise and measurement errors.

The hardware job ID is recorded for traceability. The experiment does not require another hardware submission to complete the current analysis.

In [12]:
from qiskit_ibm_runtime.executor_sampler import Sampler

# Safety checks before submission
assert backend.status().operational, "IBM backend is not operational"
assert qaoa_isa_adv.num_parameters == 0, "Circuit has unbound parameters"
assert qaoa_isa_adv.num_clbits >= 20, "Circuit is missing measurements"

SHOTS_ADV = 256

print("ADVANCED QAOA — REAL HARDWARE RUN")
print("=" * 45)
print("Backend:", backend.name)
print("Logical qubits:", qc_advanced.num_qubits)
print("Compiled circuit depth:", qaoa_isa_adv.depth())
print("CZ gates:", qaoa_isa_adv.count_ops().get("cz", 0))
print("Shots:", SHOTS_ADV)

# Submit one job to IBM hardware
sampler_adv = Sampler(mode=backend)
job_adv = sampler_adv.run([qaoa_isa_adv], shots=SHOTS_ADV)

hardware_job_id_adv = job_adv.job_id()
print("\nHardware job ID:", hardware_job_id_adv)
print("Initial job status:", job_adv.status())

# Retrieve results
result_adv = job_adv.result()
counts_adv = result_adv[0].data.meas.get_counts()

print("\nHardware measurement counts:")
print(counts_adv)
print("Total shots:", sum(counts_adv.values()))

qiskit_runtime_service.backends:WARNING:2026-10-14 12:25:46,611: Using instance: open-instance, plan: open


ADVANCED QAOA — REAL HARDWARE RUN
Backend: ibm_kingston
Logical qubits: 20
Compiled circuit depth: 680
CZ gates: 938
Shots: 256

Hardware job ID: db7n9iuil99s73fogb5g
Initial job status: QUEUED

Hardware measurement counts:
{'00001011011100111101': 1, '10110011100000000100': 1, '00001011100010011001': 1, '10110001011101111110': 1, '00010001000100111001': 1, '11010000101101110011': 1, '11011111000110010001': 1, '11010001000010001001': 1, '01010011011011001001': 1, '10011101101001110110': 1, '00000100110110000010': 1, '00001100000011111111': 1, '00011110010011111111': 1, '00010110110001000100': 1, '11110001101100000100': 1, '11101101100101100100': 1, '11011100111000101001': 1, '11111010010110111011': 1, '00000000100000000010': 1, '11001011111001101000': 1, '01000001101011001000': 1, '10010000111101000000': 1, '10010010000100011110': 1, '10100111101110000101': 1, '10111000110011100011': 1, '01001100000101100011': 1, '10001100110111000011': 1, '00000101111101001001': 1, '101100000001110011

In [13]:
print("Advanced QAOA job:", hardware_job_id_adv)

print("\nFinal job status:")
print(job_adv.status())

print("\nQPU usage:")
try:
    print(job_adv.usage(), "seconds")
except Exception as e:
    print("Usage not available yet:", e)

print("\nJob metrics:")
try:
    print(job_adv.metrics())
except Exception as e:
    print("Metrics not available:", e)

Advanced QAOA job: db7n9iuil99s73fogb5g

Final job status:
DONE

QPU usage:
2 seconds

Job metrics:
{'caller': 'qiskit', 'qiskit_version': 'qiskit_ibm_runtime-0.50.0,qiskit-2.5.2*,qiskit_aer-0.17.2*', 'timestamps': {'created': '2026-10-14T12:25:47.566064Z', 'finished': '2026-10-14T12:25:53.377106Z', 'running': '2026-10-14T12:25:48.969537Z'}, 'usage': {'qpu_charge_time_seconds': 2, 'value': 0.0296, 'unit': 'resource_units', 'status': 'complete', 'details': [{'value': 0.0296, 'unit': 'resource_units', 'metric': {'id': 'caffe3ca-729c-5750-8777-df63bbf8bfd9', 'value': 2}}]}, 'circuits_execution_time_ns': 72139520}


### Step 11: Decode and Evaluate the Hardware Results

The returned bitstrings are converted into binary stock-selection vectors. Each candidate is checked for exactly 10 selected stocks and the sector constraints before being accepted as a valid portfolio.

For valid samples, the notebook calculates the unpenalized risk-return objective and compares the best observed portfolio with the exact classical reference solution.

The hardware run produced 1 valid portfolio out of 256 shots, corresponding to a valid-sample rate of 0.39%. The exact classical optimum was not observed. The best valid sampled portfolio had a cost of −0.794773, compared with −1.111739 for the classical optimum, giving an absolute cost gap of 0.316966.

Only one valid sample was observed, so these results provide limited evidence about the distribution of feasible solutions. The observed difference from simulation is consistent with possible effects of circuit depth, hardware noise, sampling variability, and parameter quality, but this experiment does not isolate the contribution of each factor.

In [14]:
import numpy as np

# Analyze the existing Advanced hardware results
K = 10
sector_cap = 2
total_shots = sum(counts_adv.values())

valid_shots = 0
optimal_shots = 0
best_valid_cost = float("inf")
best_valid_stocks = None
unique_valid_portfolios = set()

classical_optimum_cost = -1.111739

for bitstring, count in counts_adv.items():
    # Restore original qubit order
    x = np.array([int(bit) for bit in bitstring[::-1]], dtype=int)

    selected = np.where(x == 1)[0]
    sector_counts = [
        int(x[start:start + 4].sum())
        for start in range(0, 20, 4)
    ]

    # Check cardinality and sector constraints
    valid = (
        len(selected) == K
        and all(s <= sector_cap for s in sector_counts)
    )

    # Original, unpenalized portfolio objective
    cost = float(q * (x @ sigma @ x) - mu @ x)

    if valid:
        valid_shots += count
        unique_valid_portfolios.add(tuple(selected))

        if cost < best_valid_cost:
            best_valid_cost = cost
            best_valid_stocks = selected.tolist()

        if np.array_equal(x, best_x):
            optimal_shots += count

print("ADVANCED QAOA — HARDWARE EVALUATION")
print("=" * 45)
print("Hardware job ID:", hardware_job_id_adv)
print("Total shots:", total_shots)
print(
    f"Valid shots: {valid_shots}/{total_shots} "
    f"({100 * valid_shots / total_shots:.2f}%)"
)
print("Unique valid portfolios:", len(unique_valid_portfolios))
print("Exact classical optimum sampled:", optimal_shots)

if best_valid_stocks is not None:
    print("\nBest valid sampled portfolio:")
    print([f"Stock {i+1}" for i in best_valid_stocks])
    print(f"Portfolio cost: {best_valid_cost:.6f}")
    print(f"Classical optimum cost: {classical_optimum_cost:.6f}")
    print(
        "Absolute cost gap:",
        f"{abs(best_valid_cost - classical_optimum_cost):.6f}"
    )
else:
    print("\nNo valid portfolios were sampled.")

ADVANCED QAOA — HARDWARE EVALUATION
Hardware job ID: db7n9iuil99s73fogb5g
Total shots: 256
Valid shots: 1/256 (0.39%)
Unique valid portfolios: 1
Exact classical optimum sampled: 0

Best valid sampled portfolio:
['Stock 1', 'Stock 4', 'Stock 7', 'Stock 8', 'Stock 10', 'Stock 11', 'Stock 13', 'Stock 14', 'Stock 17', 'Stock 19']
Portfolio cost: -0.794773
Classical optimum cost: -1.111739
Absolute cost gap: 0.316966


### Advanced QAOA: IBM Quantum Hardware Evaluation

**Hardware:** IBM `ibm_kingston`  
**Job ID:** `db7n9iuil99s73fogb5g`  
**Circuit:** 20 logical qubits, compiled circuit depth 680  
**Shots:** 256

The hardware experiment produced 1 valid portfolio out of 256 measurements (0.39%). The exact classical optimum was not observed. The best valid sampled portfolio was Stock 1, Stock 4, Stock 7, Stock 8, Stock 10, Stock 11, Stock 13, Stock 14, Stock 17, and Stock 19, with an objective cost of −0.794773.

The exhaustive classical reference solution had a cost of −1.111739, giving an absolute cost gap of 0.316966. Compared with the simulator's 10.30% valid-sample rate, the hardware's valid-sample rate was substantially lower.

Possible contributing factors include circuit depth, hardware noise, strict portfolio constraints, limited shots, and pilot parameters that were not shown to have converged. These results demonstrate the challenges of executing constrained QAOA on current quantum hardware; they do not demonstrate quantum advantage.

**Conclusion:** The classical exhaustive search remains the verified reference optimum for this instance. The hardware experiment is a useful evaluation of QAOA's practical limitations.